# fase 5: preparar x e y

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("../data/processed/hourly_features.csv")
df.shape

## 1. split temporal

treino: janeiro a maio de 2026. teste: junho de 2026. corte por hour_utc (nao por train_test_split aleatorio) porque o modelo tem que prever o futuro a partir do passado -- um split aleatorio deixaria o treino "ver" horas depois das que ele vai prever.

In [ ]:
hour_utc = pd.to_datetime(df["hour_utc"], utc=True)
cutoff = pd.Timestamp("2026-06-01", tz="UTC")  # corte em hour_utc, nao no mes local, pra nao ambiguar a fronteira

train_mask = hour_utc < cutoff
test_mask = hour_utc >= cutoff

## 2. features e alvo

excluidas, com motivo:
- avg_bikes: vem da mesma coluna do alvo, vazamento
- empty_share: idem, vazamento
- readings_count: so existe depois que a hora fechou, nao e conhecido no momento da previsao
- month: treino vai so ate maio, teste e junho -- o modelo nunca veria month=6 no treino
- name: duplica station_id
- hour_utc: e a chave do split, nao feature

In [ ]:
FEATURE_COLUMNS = [
    "hour_of_day", "day_of_week", "is_weekend",
    "temp", "prcp", "rhum", "wspd",
    "capacity", "lat", "lon",
    "station_id",
]
TARGET_COLUMN = "is_empty"

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

## 3. pipeline de pre-processamento

rhum tem 345 nulos, todos no periodo de treino. a mediana pra preencher tem que vir SO do treino (fit em X_train) -- se calculasse na base inteira, o treino "veria" informacao de junho atraves da mediana.

station_id vira one-hot (115 categorias). handle_unknown="ignore" e seguranca padrao de split temporal: se uma estacao so existisse no teste, ela virava zeros em vez de quebrar o transform (nao e o caso aqui, as 115 aparecem nos dois lados).

In [ ]:
numeric_features = ["hour_of_day", "day_of_week", "is_weekend", "temp", "prcp", "wspd", "capacity", "lat", "lon"]
rhum_feature = ["rhum"]
categorical_features = ["station_id"]

preprocessor = ColumnTransformer(transformers=[
    ("numeric", "passthrough", numeric_features),
    ("rhum", SimpleImputer(strategy="median"), rhum_feature),
    ("station", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

X_train_transformed = pipeline.fit_transform(X_train)  # fit so no treino
X_test_transformed = pipeline.transform(X_test)  # teste so transforma, nunca re-fita

## 4. resumo

In [ ]:
print("shape treino:", X_train.shape)
print("shape teste:", X_test.shape)
print("taxa de is_empty no treino:", round(y_train.mean(), 4))
print("taxa de is_empty no teste:", round(y_test.mean(), 4))
print("colunas depois do one-hot:", X_train_transformed.shape[1])